# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

In [1]:
!pip uninstall -y torch torchvision torchaudio transformers protobuf tensorflow tensorflow-cpu
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q vllm==0.8.5 transformers==4.51.3 accelerate bitsandbytes tqdm sympy antlr4-python3-runtime==4.11.1
!pip install -q protobuf==4.25.3

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: protobuf 5.29.6
Uninstalling protobuf-5.29.6:
  Successfully uninstalled protobuf-5.29.6
Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 116.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 

In [1]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0))
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
print("ALL IMPORTS OK")

torch: 2.6.0+cu124
cuda: 12.4
gpu: NVIDIA A100-SXM4-80GB
INFO 05-31 18:20:17 [__init__.py:239] Automatically detected platform cuda.
ALL IMPORTS OK


## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [2]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment.

In [3]:
# activate venv after installation. This needs to be run everytime.
# !source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
os.listdir('/content/drive/MyDrive/151B_SP26_Competition-main/data')

['public.jsonl']

In [6]:
import torch
torch.cuda.get_device_name(0)

'NVIDIA A100-SXM4-80GB'

In [7]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES (Changed from 1 to 0 for Colab)
DATA_PATH   = "/content/drive/MyDrive/151B_SP26_Competition-main/data/public.jsonl"
OUTPUT_PATH = "/content/drive/MyDrive/151B_SP26_Competition-main/results/currentt_results.jsonl"
MAX_TOKENS  = 8192 #32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [8]:
import os
os.chdir('/content/drive/MyDrive/151B_SP26_Competition-main')

In [9]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [10]:
# multiple_answers variant
SYSTEM_PROMPT_MATH = (
    "Solve the math problem. Show only the necessary reasoning. "
    "Final answer rules:"
    "- Include EVERY requested answer in the final answer."
    "- If the problem has multiple parts or multiple blanks, put all answers in ONE final \\boxed{...}, separated by commas."
    "- Use exact form."
    "- No decimal approximations."
    "- Keep expressions symbolic."
    "- Use \\frac{}{}, powers, \\sqrt{}, \\pi, \\ln{}, \\arctan{} when appropriate."
    "- Only use decimals for numbers that are already decimals in the problem."
    "- End with the final answer in \\boxed{...}."
)

SYSTEM_PROMPT_MCQ = (
    "Solve the multiple-choice math problem. "
    "Output ONLY the correct choice letter inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [11]:
#!pip install -q vllm==0.9.0

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.80,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

# MCQ: majority vote with n=3, Free-form: single generation
sampling_params_mcq = SamplingParams(
    n = 3,
    max_tokens = MAX_TOKENS,
    temperature = 0.6,
    top_p = 0.95,
    top_k = 20,
    min_p = 0.0,
    presence_penalty = 0.0,
    repetition_penalty = 1.0,
)

sampling_params_free = SamplingParams(
    n = 1,
    max_tokens = MAX_TOKENS,
    temperature = 0.6,
    top_p = 0.95,
    top_k = 20,
    min_p = 0.0,
    presence_penalty = 0.0,
    repetition_penalty = 1.0,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

INFO 05-31 18:21:30 [config.py:717] This model supports multiple tasks: {'score', 'generate', 'classify', 'embed', 'reward'}. Defaulting to 'generate'.
INFO 05-31 18:21:30 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=32768.


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

WARNING 05-31 18:21:33 [utils.py:2382] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/getting_started/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized
INFO 05-31 18:24:40 [core_client.py:439] Core engine process 0 ready.
Model loaded.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [13]:
# !pip install -U bitsandbytes

In [14]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = 'left'

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [15]:
import re
from collections import Counter

def extract_boxed_answer(text: str) -> str:
    matches = list(re.finditer(r'\\boxed\{', text))
    if not matches:
        return "EXTRACTION_FAILED"
    last_match = matches[-1]
    start_idx = last_match.end()
    brace_count = 1
    for i in range(start_idx, len(text)):
        if text[i] == '{':
            brace_count += 1
        elif text[i] == '}':
            brace_count -= 1
        if brace_count == 0:
            return text[start_idx:i].strip()
    return "EXTRACTION_FAILED"

def get_majority_vote(generations):
    extracted = [extract_boxed_answer(t) for t in generations]
    valid = [a for a in extracted if a != "EXTRACTION_FAILED"]
    if not valid:
        return "NO_VALID_ANSWERS"
    return Counter(valid).most_common(1)[0][0]

# Run full dataset
N = len(data)
prompts = []
is_mcq_list = []

for item in data:
    is_mcq = bool(item.get("options"))
    is_mcq_list.append(is_mcq)
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Use different sampling params per question type
params_list = [sampling_params_mcq if is_mcq else sampling_params_free for is_mcq in is_mcq_list]

print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=params_list)

responses = []
for i, out in enumerate(outputs):
    if is_mcq_list[i]:
        generations = [seq.text for seq in out.outputs]
        winning_answer = get_majority_vote(generations)
        responses.append(winning_answer)
    else:
        single = out.outputs[0].text
        answer = extract_boxed_answer(single)
        responses.append(single if answer == "EXTRACTION_FAILED" else single)

for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 1126 questions...


Processed prompts:   0%|          | 0/1876 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…


── Response 0 (id=0) ──
Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me make sure I know what the first few even whole numbers are. The positive even whole numbers start from 2, right? So the first one is 2, the second is 4, the third is 6, and so on. So it's an arithmetic sequence where each term increases by 2. 

I remember that the sum of the first n terms of an ...

── Response 1 (id=1) ──
B 

── Response 2 (id=2) ──
This is a complex or challenging question, and it is difficult to provide a direct and correct answer. I need to think about it.
Well, so this is a Newton's Law of Cooling problem, right? Let me recall what that says. Newton's Law of Cooling states that the rate of change of the temperature of an object is proportional to the difference between its temperature and the ambient temperature. So, math ...


### Generate with Transformers (for Datahub)

In [16]:
# import torch

# print("CUDA available:", torch.cuda.is_available())

# if torch.cuda.is_available():
#     print("GPU:", torch.cuda.get_device_name(0))
# else:
#     print("No GPU detected")

In [17]:
# MAX_NEW_TOKENS = 4096
# N = 10

# responses = []
# for item in data[:N]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     inputs = tokenizer(prompt_text, return_tensors="pt").to(llm.device)
#     with torch.no_grad():
#         output_ids = llm.generate(
#             **inputs,
#             max_new_tokens=MAX_NEW_TOKENS,
#             temperature=0.6,
#             top_p=0.95,
#             top_k=20,
#             do_sample=True,
#         )
#     new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
#     print(f"Done {len(responses)}/{N}")

In [18]:
# # The whole dataset in batch sizes of 5

# MAX_TOKENS = 2048
# BATCH_SIZE = 2

# responses = []

# print(f"Generating responses for {len(data)} questions in batches of {BATCH_SIZE}...")

# for start in range(0, len(data), BATCH_SIZE):

#     batch = data[start:start+BATCH_SIZE]

#     # Build prompts
#     prompts = []
#     for item in batch:
#         system, user = build_prompt(item["question"], item.get("options"))

#         prompt_text = tokenizer.apply_chat_template(
#             [{"role": "system", "content": system},
#              {"role": "user",   "content": user}],
#             tokenize=False,
#             add_generation_prompt=True,
#         )

#         prompts.append(prompt_text)

#     # Tokenize batch
#     inputs = tokenizer(
#         prompts,
#         return_tensors="pt",
#         padding=True,
#         truncation=True,
#         max_length=2048,
#     ).to(llm.device)

#     # Generate
#     with torch.no_grad():
#         output_ids = llm.generate(
#             **inputs,
#             max_new_tokens=MAX_TOKENS,
#             temperature=0.6,
#             top_p=0.95,
#             top_k=20,
#             repetition_penalty=1.0,
#             do_sample=True,
#             pad_token_id=tokenizer.eos_token_id,
#         )

#     # Decode
#     for i, out in enumerate(output_ids):
#         new_tokens = out[inputs["input_ids"].shape[1]:]

#         text = tokenizer.decode(
#             new_tokens,
#             skip_special_tokens=True
#         ).strip()

#         responses.append(text)

#     # Free GPU memory between batches
#     del inputs, output_ids
#     torch.cuda.empty_cache()

#     print(f"Completed {min(start+BATCH_SIZE, len(data))}/{len(data)}")

# # Preview first few outputs
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [19]:
#!pip install antlr4-python3-runtime==4.11

In [20]:
def extract_letter(text: str) -> str:
    # Handle case where response is already just a letter
    if len(text.strip()) == 1 and text.strip().upper() in "ABCDEFGHIJ":
        return text.strip().upper()
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
import sys
sys.path.insert(0, "/content/drive/MyDrive/151B_SP26_Competition-main")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 1126/1126 [01:03<00:00, 17.87it/s]

Scoring complete. 1126 results.


## 8. Summary

Print accuracy broken down by question type.

In [21]:
## Comparison: Baseline vs Current

import json

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

def split(data):
    return [r for r in data if r["is_mcq"]], [r for r in data if not r["is_mcq"]]

baseline = [json.loads(line) for line in open("/content/drive/MyDrive/151B_SP26_Competition-main/results/starter_results.jsonl")]
current = results

b_mcq, b_free = split(baseline)
c_mcq, c_free = split(current)

print("=" * 55)
print(f"{'':20} {'Baseline':>12} {'Current':>12} {'Delta':>8}")
print("=" * 55)
print(f"{'MCQ':20} {acc(b_mcq):>11.2f}% {acc(c_mcq):>11.2f}% {acc(c_mcq)-acc(b_mcq):>+7.2f}%")
print(f"{'Free-form':20} {acc(b_free):>11.2f}% {acc(c_free):>11.2f}% {acc(c_free)-acc(b_free):>+7.2f}%")
print(f"{'Overall':20} {acc(baseline):>11.2f}% {acc(current):>11.2f}% {acc(current)-acc(baseline):>+7.2f}%")
print("=" * 55)

                         Baseline      Current    Delta
MCQ                        71.47%       54.40%  -17.07%
Free-form                  54.59%       58.19%   +3.60%
Overall                    60.21%       56.93%   -3.29%


In [22]:
from huggingface_hub import list_repo_files
for f in list_repo_files("dcraver2005/r8_a16_numinamath_16bit"):
    print(f)

.gitattributes
README.md
added_tokens.json
chat_template.jinja
config.json
generation_config.json
merges.txt
model-00001-of-00002.safetensors
model-00002-of-00002.safetensors
model.safetensors.index.json
special_tokens_map.json
tokenizer.json
tokenizer_config.json
vocab.json


In [23]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok = AutoTokenizer.from_pretrained("dcraver2005/r8_a16_numinamath_16bit")
m = AutoModelForCausalLM.from_pretrained("dcraver2005/r8_a16_numinamath_16bit", torch_dtype=torch.float16, device_map="auto")
messages = [{"role": "user", "content": "What is 2+2?"}]
text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(text, return_tensors="pt").to("cuda")
out = m.generate(**inputs, max_new_tokens=200)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

</think>

1. **Understanding the Question**:
   The question asks for the result of the arithmetic operation $2+2$.

2. **Performing the Addition**:
   According to basic arithmetic rules, adding two numbers means finding their sum. Here, we are adding the number $2$ to itself.

   \[
   2 + 2 = 4
   \]

3. **Conclusion**:
   The sum of $2$ and $2$ is $4$. Therefore, the final answer is:

   \[
   \boxed{4}
   \]


In [24]:
print(responses[:5])
print(is_mcq_list[:5])

['Okay, let\'s see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me make sure I know what the first few even whole numbers are. The positive even whole numbers start from 2, right? So the first one is 2, the second is 4, the third is 6, and so on. So it\'s an arithmetic sequence where each term increases by 2. \n\nI remember that the sum of the first n terms of an arithmetic sequence can be found using the formula: S_n = n/2 * (a_1 + a_n), where a_1 is the first term and a_n is the nth term. Alternatively, another formula is S_n = n * a_1 + n(n - 1)/2 * d, where d is the common difference. Let me confirm which one is better here.\n\nFirst, let\'s figure out what the first term a_1 is. The first positive even whole number is 2, so a_1 = 2. The common difference d is 2, since each term is 2 more than the previous. We need the sum of the first 325 terms, so n = 325.\n\nLet me try the first formula: S_n = n/2 * (a_1 + a_n). So I need to find a_n, the

In [25]:
print(Counter(responses[:50]))
no_valid = sum(1 for r in responses if r == "NO_VALID_ANSWERS")
print(f"NO_VALID_ANSWERS: {no_valid}")

Counter({'NO_VALID_ANSWERS': 5, 'B': 2, 'E': 2, 'Okay, let\'s see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me make sure I know what the first few even whole numbers are. The positive even whole numbers start from 2, right? So the first one is 2, the second is 4, the third is 6, and so on. So it\'s an arithmetic sequence where each term increases by 2. \n\nI remember that the sum of the first n terms of an arithmetic sequence can be found using the formula: S_n = n/2 * (a_1 + a_n), where a_1 is the first term and a_n is the nth term. Alternatively, another formula is S_n = n * a_1 + n(n - 1)/2 * d, where d is the common difference. Let me confirm which one is better here.\n\nFirst, let\'s figure out what the first term a_1 is. The first positive even whole number is 2, so a_1 = 2. The common difference d is 2, since each term is 2 more than the previous. We need the sum of the first 325 terms, so n = 325.\n\nLet me try the first formula: S_n 

In [26]:
# Check first 5 MCQ results
for r in [r for r in results if r['is_mcq']][:5]:
    print(f"Gold: {r['gold']}, Response: {r['response']}, Correct: {r['correct']}")

Gold: F, Response: B, Correct: False
Gold: C, Response: C, Correct: True
Gold: A, Response: A, Correct: True
Gold: E, Response: NO_VALID_ANSWERS, Correct: False
Gold: G, Response: NO_VALID_ANSWERS, Correct: False


In [27]:
for r in [r for r in results if r['is_mcq']][:3]:
    print(f"Gold: {r['gold']}")
    print(f"Response: {r['response']}")
    print("---")

Gold: F
Response: B
---
Gold: C
Response: C
---
Gold: A
Response: A
---


In [28]:
import os
print(os.listdir('/content/drive/MyDrive/151B_SP26_Competition-main/results'))

['starter_results.jsonl', 'sft_results.jsonl', 'r8_a16_30k_results.jsonl']


In [29]:
# print(MODEL_ID)

In [30]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok = AutoTokenizer.from_pretrained(MODEL_ID)
m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
inputs = tok("What is 2+2?", return_tensors="pt").to("cuda")
out = m.generate(**inputs, max_new_tokens=100)
print(tok.decode(out[0]))

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

What is 2+2? Is it 4 or 5? What is the correct answer? - Quora
What is 2+2? Is it 4 or 5? What is the correct answer?
Ad by TruthFinder
Have you ever googled a question with the answer "I don't know"?
This search engine reveals so much more. Type in your question
(Continue reading)
2 Answers
Best
Amit Kumar
, B.Tech from Indian Institute of Technology (202


In [31]:
print(responses[0])

Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me make sure I know what the first few even whole numbers are. The positive even whole numbers start from 2, right? So the first one is 2, the second is 4, the third is 6, and so on. So it's an arithmetic sequence where each term increases by 2. 

I remember that the sum of the first n terms of an arithmetic sequence can be found using the formula: S_n = n/2 * (a_1 + a_n), where a_1 is the first term and a_n is the nth term. Alternatively, another formula is S_n = n * a_1 + n(n - 1)/2 * d, where d is the common difference. Let me confirm which one is better here.

First, let's figure out what the first term a_1 is. The first positive even whole number is 2, so a_1 = 2. The common difference d is 2, since each term is 2 more than the previous. We need the sum of the first 325 terms, so n = 325.

Let me try the first formula: S_n = n/2 * (a_1 + a_n). So I need to find a_n, the 325th term

In [32]:
print(results[0])

{'id': 0, 'is_mcq': False, 'gold': ['325*(1+325)'], 'response': 'Okay, let\'s see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me make sure I know what the first few even whole numbers are. The positive even whole numbers start from 2, right? So the first one is 2, the second is 4, the third is 6, and so on. So it\'s an arithmetic sequence where each term increases by 2. \n\nI remember that the sum of the first n terms of an arithmetic sequence can be found using the formula: S_n = n/2 * (a_1 + a_n), where a_1 is the first term and a_n is the nth term. Alternatively, another formula is S_n = n * a_1 + n(n - 1)/2 * d, where d is the common difference. Let me confirm which one is better here.\n\nFirst, let\'s figure out what the first term a_1 is. The first positive even whole number is 2, so a_1 = 2. The common difference d is 2, since each term is 2 more than the previous. We need the sum of the first 325 terms, so n = 325.\n\nLet me try the fir

In [33]:
print(data[0])
print(responses[0][:200])

{'question': 'Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]', 'answer': ['325*(1+325)'], 'id': 0}
Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me make sure I know what the first few even whole numbers are. The positive even whole numbers sta


In [34]:
import re
for r in results[:5]:
    print(r['correct'], re.search(r'\\boxed\{([A-Za-z])\}', r['response']))

True None
False None
False None
True None
True None


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [35]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 1126 records to /content/drive/MyDrive/151B_SP26_Competition-main/results/currentt_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!